In [0]:
# Aggregate and Join for KPI Zero Coverage Percentage Monthly

from pyspark.sql.functions import year, month, count, sum, col

# Aggregate metrics
table1 = fact_df.groupBy(
    "payer_id",
    year(col("start")).alias("year"),
    month(col("start")).alias("month")
).agg(
    count("*").alias("total"),
    sum("has_payer_coverage").alias("covered_by_payer")
).withColumn(
    "covered_percentage",
    (col("covered_by_payer") / col("total")) * 100
)

# Join with payer dimension
table2 = table1.join(
    spark.table("medical_project.gold.dim_payer"),
    "payer_id",
    "inner"
).select(
    col("payer_id"),
    col("payer_name"),
    col("year"),
    col("month"),
    col("total"),
    col("covered_by_payer"),
    col("covered_percentage")
).orderBy("year", "month")

# Save as table
table2.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_zero_payer_coverage")

In [0]:
display(table2)